# 👑 04. Supervisor 패턴 & 멀티에이전트 오케스트레이션 (5대 협업 프로토콜 & Blackboard Pattern)

단일 에이전트(Single Agent)가 복잡한 시장 분석이나 데이터 수집 작업을 수행하면, 긴 작업 과정 중 컨텍스트 누적으로 **주의력 분산(Attention Drift)** 과 **토큰 비용 폭증(Context Bloat)** 이 발생합니다.

이를 해결하기 위해 에이전트 시스템은 전체 기획을 총괄하는 **Supervisor(부모)**와 특정 전문 작업만 격리 수행하는 **Worker(자식)**로 분리된 **멀티에이전트 아키텍처**를 사용합니다.

우리는 부모와 자식 간의 **정보 불균형(Information Asymmetry)**을 원천 차단하는 **5대 협업 전략**과 디스크 칠판 기반의 **Planning & Task Board 도구 엔진**을 직접 구현하고 실전 시나리오를 검증합니다.

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 핵심 내용 |
| :---:|:---|:---|
| **Step 0** | **환경 세팅** | 루트 경로 탐색, `.env` 로드, `app.tools`, `app.middleware` 임포트 |
| **Part 1** | **멀티에이전트 오케스트레이션과 5대 협업 프로토콜** | 오케스트레이션 3대 패턴, Supervisor 위임 방식, 정보 불균형과 5대 협업 프로토콜 |
| **Part 2** | **Planning & Task Board 도구 엔진** | `enter_plan` ➔ `exit_plan`(계획 확정·게시) 및 `task_create`/`task_list`/`task_update`를 통한 공유 칠판 통제 |
| **Part 3** | **Sub-Agent 래핑 & 3-Layer Protocol Overlay** | 5대 프로토콜의 코드 구현체: `3-Layer Protocol Overlay` & `invoke_sub_agent` 명세 |
| **Part 4** | **Supervisor 조립 & [시나리오 1] 가성비 분석** | 노트북 시장 데이터(`laptops_market_data.json`) 분석 & 추천 리포트 발행 (Happy Path) |
| **Part 5** | **장애 복구 & [시나리오 2] Backtracking** | `[BLOCKER]` 발생 시 이전 단계로 되돌아가 계획을 수정하는 자가 치유(Self-Healing) |
| **Part 6** | **실무 요약 & AAWS 프로덕션 확장 로드맵** | 멀티에이전트 설계 3대 황금률 및 `Scraper`/`Analyst` 파이프라인 확장 가이드 |

## 🛠️ Step 0. 환경 세팅

필요한 패키지를 로드하고 프로젝트 루트 경로 및 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.


In [9]:
import os
import sys
import json
import time
import asyncio
import nest_asyncio
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# 1. 환경변수 로드
load_dotenv(override=True)

# 2. 프로젝트 루트 경로 자동 설정 (상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Working Directory: {os.getcwd()}")
print(f"✅ Project Root: {project_root}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. LangChain 핵심 모듈 및 검증된 공용 도구(common.py, plan.py) 임포트
from pydantic import BaseModel, Field
from app.utils import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from app.utils import normalize_content

# app/tools 프로덕션 도구 및 app/middleware 계층형 시각화 미들웨어 임포트
from app.tools.common import file_read, file_writer, file_edit, grep_search, glob_search
from app.tools.plan import (
    enter_plan, exit_plan, task_create, task_list, task_update,
    check_plan_mode_tool_allowed, get_plan_history,
)
from app.middleware import HierarchicalVisualizerMiddleware

print("✅ Core Modules, Production Tools (common + plan) & Visualizer Middleware Imported Successfully!")


✅ Working Directory: /workspaces/AAWS
✅ Project Root: /workspaces/AAWS
✅ Core Modules, Production Tools (common + plan) & Visualizer Middleware Imported Successfully!


---
## 📌 Part 1. 멀티에이전트 오케스트레이션과 정보 불균형(Information Asymmetry)

### 1.1 멀티에이전트 오케스트레이션(Orchestration)의 3대 패턴

복잡한 작업을 여러 에이전트가 나누어 처리할 때, **"누가 다음 작업 순서와 제어권(Control Flow)을 결정하는가?"**에 따라 크게 세 가지 오케스트레이션 패턴으로 나뉩니다.

| 구분 | 1. Round Robin (순환형) | 2. Swarm (자율 네트워크형) | 3. Supervisor (중앙 집중형) |
| :---: | :--- | :--- | :--- |
| **구조 도식** | <pre>  ┌────> (1) ────┐<br>  │              ▼<br>  │             (2)<br>  │              │<br> (3) <───────────┘<br> [종료 조건까지 순환]</pre> | <pre>  (A) <───> (B)<br>   ▲  ╲   ╱  ▲<br>   │   ╲ ╱   │<br>   │   ╱ ╲   │<br>   ▼  ╱   ╲  ▼<br>  (C) <───> (D)</pre> | <pre>       [USER]<br>         ▲<br>         ▼<br>    (Supervisor)<br>   ┌─────┼─────┐<br>   ▲     ▲     ▲<br>   ▼     ▼     ▼<br> (W1)  (W2)  (W3)</pre> |
| **제어 주체** | **사전 정의된 규칙 (Rule)** | **개별 에이전트 (Decentralized)** | **중앙 관리자 (Centralized)** |
| **동작 방식** | 정해진 순서(1➔2➔3)대로 에이전트가 돌아가며 발언/작업 수행 | 각 에이전트가 상황에 따라 다음 작업을 수행할 에이전트를 자율 지목 | 중앙의 **Supervisor**가 전체 계획을 세우고 하위 에이전트에게 작업을 할당 및 취합 |
| **장점** | 구조가 단순하며 다자간 토론이나 교차 검증 루프에 적합 | 에이전트 간 자율적인 작업 위임으로 동적인 상황에 유연하게 대처 | **Spoke-and-Hub** 구조로 흐름 제어가 명확하고, **안정적이며 구현과 디버깅이 가장 쉬움** |
| **한계 및 특징** | 명확한 **종료 조건** 설계가 필수적이며, 불필요한 턴 소모 발생 가능 | 에이전트 간 무한 핑퐁(Loop)에 빠지기 쉽고 전체 상태 추적이 어려움 | 실무 프로덕션에서 **가장 널리 사용되는 표준(De-facto Standard) 패턴** |

---

### 1.2 Supervisor 패턴과 두 가지 작업 위임(Delegation) 방식

실무에서 **Supervisor 패턴**이 가장 많이 사용되는 이유는 명확합니다. 사용자와의 소통 창구와 전체 작업의 진행 상태(State)를 중앙의 Supervisor가 독점하여 통제하므로, **시스템의 예측 가능성(Predictability)과 안정성이 가장 높고 구현이 직관적**이기 때문입니다.

이 Supervisor 패턴은 하위 에이전트에게 작업을 위임하는 방식에 따라 다시 **Agent-as-Tool (Subagent)** 방식과 **Handoff** 방식으로 나뉩니다.

| 구분 | 1. Agent-as-Tool (Subagent 캡슐화) | 2. Handoff (제어권 이양) |
| :---: | :--- | :--- |
| **구조 도식** | <pre>        [USER]<br>          ▲<br>          │  (Supervisor만 소통)<br>          ▼<br>     (Supervisor)<br>   ┌──────┼──────┐<br>   ▲      ▲      ▲  (도구 호출 및<br>   ▼      ▼      ▼   결과 반환)<br>[(W1)] [(W2)] [(W3)]<br>* 함수(Tool)로 하위 에이전트 캡슐화</pre> | <pre> ┌──────>[USER]<br> │         │  (최초 요청)<br> │         ▼<br> │    (Supervisor)<br> │  ┌──────┼──────┐<br> │  ▲      ▲      ▲  (위임 및<br> │  ▼      ▼      ▼   Supervisor로 반환)<br> └─>(W1)  (W2)   (W3)<br>* 작업 중 User에게 직접 질의(HITL) 가능</pre> |
| **핵심 개념** | 하위 에이전트를 **하나의 함수(Tool)로 캡슐화**하여 호출하고 결과만 반환받음 | 하위 에이전트에게 **활성 제어권(Active Control)을 잠시 위임**했다가 완료 시 **Supervisor에게 반환** |
| **컨텍스트** | **컨텍스트 격리 (Context Isolation)**<br>하위 에이전트는 독립된 샌드박스에서 실행되어 부모의 컨텍스트가 오염되지 않음 | **컨텍스트 공유 (Context Sharing)**<br>이전 대화 히스토리를 공유받아 맥락 유실 없이 작업 수행 |
| **제어권 및 특징** | **Supervisor가 제어권 독점 유지**<br>하위 에이전트는 User와 직접 대화할 수 없으며, 오직 Supervisor에게만 결과 보고 | **유연한 제어 (Human-in-the-Loop)**<br>제어권을 위임받은 하위 에이전트가 작업에 필요한 정보를 **User에게 직접 물어볼 수 있음** |

---

### 1.3 거시적 시야 vs 미시적 시야: '정보 불균형'의 발생과 2가지 해결 접근법

복잡한 데이터 수집이나 장기 분석 작업에서는 하위 에이전트의 방대한 시행착오 로그(HTML 원본, 쉘 출력 등)로부터 메인 대화창을 보호하기 위해 **컨텍스트가 격리된 `Agent-as-Tool (Subagent)` 방식**을 주로 채택합니다.

하지만 컨텍스트를 격리하는 순간, 필연적으로 **부모 에이전트(Supervisor)와 자식 에이전트(Subagent) 간의 정보 불균형(Information Asymmetry)** 문제가 발생합니다.
* **부모 에이전트 (거시적 정보 중심)**: 사용자의 최종 요구사항, 전체 작업 계획, 이전 단계까지의 진행 맥락 등 **거시적(Macro) 정보**에 치중합니다.
* **자식 에이전트 (미시적 정보 중심)**: 전체 맥락을 모르는 상태에서 격리된 채로 깨어나, 당장 눈앞의 파일 읽기, 스크래핑, 코드 실행 등 **미시적(Micro) 정보**만 취급합니다.

이로 인해 부모와 자식 간에 정보가 비효율적으로 전달되거나 컨텍스트가 뒤섞이면 다음과 같은 **4대 병목 현상**이 나타납니다:
1. **Context Bloat & Token Explosion (컨텍스트 오염 및 비용 폭증)**: 하위 에이전트의 원본 HTML, 수천 줄의 raw 데이터, 중간 오류 로그가 상위 오케스트레이터 대화창에 누적되면 토큰 비용이 기하급수적으로 증가합니다.
2. **Attention Drift & Hallucination (주의력 분산 및 환각 유발)**: 컨텍스트가 비대해지면 LLM의 주의 집중도가 분산되어 사용자의 최초 핵심 요구사항(가성비 산출 공식, 제약 조건 등)을 망각합니다.
3. **Black-box State Trap (블랙박스 상태 관리의 한계)**: 런타임 진행 상태를 보이지 않는 메모리(`state["messages"]`)로만 주고받으면, 어느 단계에서 왜 실패했는지 추적/디버깅이 불가능해집니다.
4. **Cascade Failure (단일 실패의 연쇄 붕괴 & Backtracking 부재)**: 하위 작업에서 파일 손상이나 사이트 차단이 발생했을 때 상위 에이전트가 유연하게 계획을 재설계(Backtracking)하지 못하면 전체 시스템이 멈춥니다.

따라서 부모가 자식에게 적절한 맥락을 전달하고 결과를 보고받기 위해 정보 불균형을 해소해야 하며, 여기에는 크게 **두 가지 접근법**이 존재합니다.

#### ① 접근법 1: 개별 Subagent 호출 도구의 최적화 (맞춤형 지시 — *과거의 방식*)
* **개념**: 각 하위 에이전트의 역할에 맞춰 전용 호출 도구와 입력 스키마(Parameter)를 개별적으로 정교하게 설계하는 방식입니다.
* **비유**: **작은 팀의 팀장**이 각 팀원의 담당 업무, 성향, 업무 스타일에 따라 **지시 방식과 제공 자료를 일일이 다르게 맞춰주는 것**과 같습니다.
* **한계**: 팀원(Subagent)이 2~3명일 때는 잘 작동하지만, 에이전트의 종류가 늘어나고 시스템이 커지면 **각 에이전트별 전용 호출 도구 및 프롬프트를 관리하는 데 엔지니어링 비용이 많이 발생합니다.**

#### ② 접근법 2: 위임 프로토콜의 일반화 (표준화된 업무 위임 — *현대적 방식*)
* **개념**: 어떤 Subagent건 안정적인 업무 위임이 가능한 **일반화된 패턴(단일 표준 호출 도구 `invoke_sub_agent`와 공통 프로토콜)**을 만들어내는 것입니다.
* **비유**: 팀원이 많아졌을 때 누구에게 업무를 맡기든 **표준 업무 지시서, 공유 칠판(Blackboard), 요약 보고서 양식**만 지키면 일관되게 위임과 보고가 돌아가도록 **표준 업무 프로토콜**을 구축하는 것과 같습니다.
* **핵심 연결**: 바로 이 **'일반화된 업무 위임 패턴'**이 아래에서 설명할 **Blackboard 아키텍처**와 **5대 협업 프로토콜**입니다.

---

### 1.4 정보 불균형 극복 아키텍처: Supervisor-Worker 구조 & Blackboard Pattern

일반화된 위임 구조를 구현하기 위해 **역할 격리(Role Isolation)**와 **공유 칠판 패턴(Blackboard Pattern)**을 결합합니다:

* **Supervisor (부모 오케스트레이터)**: 사용자의 거시적 목표를 분석하여 계획을 수립하고, 하위 태스크를 디스패치하며, 최종 결과를 종합 보고합니다.
* **Worker (자식 전문 에이전트)**: 부모의 대화 기록에 일절 간섭받지 않는 격리된 샌드박스에서 도구를 자율 연쇄 실행하여 주어진 일감만 완수합니다.
* **Blackboard (공유 칠판)**: 에이전트 간 메모리를 직접 침범하지 않고, 파일 시스템 상의 구조화된 상태(`plan_state.json`, `task_state.json`)와 산출물 디렉토리(`artifacts/`)를 **단일 진실 공급원(SSOT)**으로 삼아 투명하게 상태를 공유합니다.

```
┌────────────────────────────────────────────────────────────────────────┐
│                        📋 Blackboard (공유 칠판)                        │
│     plan_state.json (거시 전략) / task_state.json (미시 체크리스트)      │
│     artifacts/ (영구 보존 산출물 저장소)                                │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ (상태 동기화 & 산출물 보존)
┌───────────────────────────────────▼────────────────────────────────────┐
│                    👑 Supervisor (Parent Orchestrator)                 │
│  - 사용자 요구사항 분석 & 칠판에 실행 계획/태스크 수립                 │
│  - ⚡ Dynamic Context Pruning (Worker에게 필요한 타겟 파일/일감만 슬라이싱)│
│  - 🔀 Backtracking (Worker의 [BLOCKER] 수신 시 계획 동적 재수립)      │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ invoke_sub_agent(target_file_list, instruction)
                                    ▼ (No Parent History Pollution)
┌────────────────────────────────────────────────────────────────────────┐
│                     👷 Worker Layer (Isolated Sandbox)                 │
│  - 🛡️ Prompt Layering (No-Chat 침묵 실행 & Fact-Only 보고 규약)        │
│  - 👶 Self-directed Gathering (인계받은 파일을 도구로 자율 탐색)       │
│  - 📄 Artifacts 디스크 생성 & [TASK REPORT] 요약 포인터만 반환          │
└────────────────────────────────────────────────────────────────────────┘
```

---

### 1.5 정보 불균형을 원천 차단하는 5대 협업 프로토콜 (Deep Dive)

| # | 프로토콜 명칭 | 핵심 원리 (Mechanism) | 해결하는 정보 불균형 문제 | 실습 구현 도구 / 코드 |
| :-: | :--- | :--- | :--- | :--- |
| **1** | **⚡ Dynamic Context Pruning**<br>(동적 맥락 가지치기) | 부모의 전체 대화 기록을 넘기지 않고, **`target_file_list`와 명확한 `task_instruction`만 슬라이싱**하여 Worker에 주입 | 자식의 Attention Drift 차단 및 토큰 낭비 제거 | `invoke_sub_agent`의 슬라이싱 인자 전달 |
| **2** | **👶 Self-directed Gathering**<br>(자율 팩트 수집) | 부모가 시시콜콜 지시하지 않고 목표만 부여 ➔ Worker가 내부 ReAct 루프에서 **도구(`file_read` 등)를 자율 연쇄 가동**하여 팩트 수집 | 상위 에이전트의 통제 오버헤드 감소 및 작업 자율성 | Worker의 `file_read`, `grep_search` 연쇄 |
| **3** | **🛡️ Prompt Layering**<br>(계층형 프롬프트 & 침묵 실행) | 부모와 자식의 Persona 분리. Worker에게 **"No-Chat(잡담/인사 금지), Fact-Only 실행 및 규격화된 포맷 출력"**을 강제 | 불필요한 대화 토큰 낭비 제거 및 응답 파싱 안정성 | Worker 전용 `worker_system_prompt` |
| **4** | **📄 Artifacts & Summary Pointer**<br>(산출물 격리 & 요약 보고) | 상세 분석 표나 대용량 데이터는 **`artifacts/` 디스크 파일로 저장**하고, 부모에게는 **5줄 `[TASK REPORT]` 요약 포인터만 반환** | 부모 컨텍스트 윈도우 오염 방지 (태스크 수십 번 반복 가능) | `file_writer` 저장 + `[TASK REPORT]` 반환 |
| **5** | **🔀 Backtracking & Self-Healing**<br>(백트래킹 자가 치유) | Worker 장애 발생 시 중단 없이 **`[BLOCKER: 원인]` 규약 반환** ➔ Supervisor가 **이전 계획 단계로 백트랙**하여 대안 데이터로 재시도 | 단일 실패의 연쇄 붕괴 방지 및 파이프라인 자가 치유 | `[BLOCKER]` 반환 ➔ 백업 카탈로그 재계획 루프 |

---
## 📌 Part 2. Planning & Task Board 도구 엔진 (Blackboard Pattern)

문자열 치환(`file_edit`) 방식은 LLM의 기억 오류로 인한 텍스트 매칭 실패 위험이 있습니다.
`app/tools/plan.py`는 Claude Code Plan Mode와 공유 칠판(Blackboard) 패턴을 결합하여, Supervisor 전용의 **구조화된 5대 Planning 도구**로 디스크 칠판(`plan_state.json`, `task_state.json`, `plan_history.json`)을 통제합니다:

1. `enter_plan(intent, goal)`: 읽기 전용 Planning Mode 진입 (`intent='new'` 시 이전 계획을 `plan_history.json`에 자동 보관·초기화, `intent='revise'` 시 기존 `BLOCKED` 태스크를 보존한 채 백트래킹 재설계 진입; `exit_plan` 전까지 `task_create` 및 `invoke_sub_agent` 호출 차단)
2. `exit_plan(plan_name, steps)`: 확정된 거시 계획을 `plan_state.json`에 게시하고 실행 잠금 해제 (하위 에이전트에 전달할 공유 파일 경로 반환)
3. `task_create`: 실행 단계에서 작업 단위 생성 (`task_state.json`에 Task ID 발급 및 `PENDING` 등록)
4. `task_list`: 활성 거시 계획(`plan_state.json`) 요약과 전체 태스크(`task_state.json`) 현황 통합 조회
5. `task_update`: 태스크 상태 갱신 (`PENDING` ➔ `IN_PROGRESS` ➔ `COMPLETED` / `BLOCKED`, 모든 태스크 완료 시 계획 상태도 `COMPLETED`로 동기화)


In [10]:
print("🛠️  app.tools.plan 5대 도구 (enter_plan ➔ exit_plan, task_create, task_list, task_update) 준비 완료!")

🛠️  app.tools.plan 5대 도구 (enter_plan ➔ exit_plan, task_create, task_list, task_update) 준비 완료!


### 📁 실습 데이터 및 산출물 보관 경로 (`artifacts/notebooks/`)
실습과 관련된 모든 원본 데이터셋은 `artifacts/notebooks/data/`에, 에이전트 생성 산출물은 **`artifacts/notebooks/generated/`**에 보관됩니다:
- `artifacts/notebooks/data/laptops_market_data.json`: 노트북 5종 스펙/가격 데이터 (시나리오 1용)
- `artifacts/notebooks/data/specs_criteria.json`: 가성비 점수 산출 기준 가이드 파일 (다중 파일 인계 실습용)
- `artifacts/notebooks/data/corrupted_catalog.bin`: 손상된 바이너리 파일 (시나리오 2 실패 유발용)
- `artifacts/notebooks/data/backup_laptops_catalog.json`: 백업 카탈로그 (시나리오 2 백트래킹용)
- `artifacts/notebooks/generated/laptop_market_report.md`: 시나리오 1에서 Worker가 생성하는 최종 마켓 리포트
- `artifacts/notebooks/generated/backup_market_report.md`: 시나리오 2 백트래킹으로 Worker가 생성하는 마켓 리포트

In [11]:
# 3. 실습용 노트북 데이터 및 아티팩트 디렉토리 초기화 (artifacts/notebooks/)
nb_workspace_dir = os.path.join(project_root, "artifacts", "notebooks")
data_dir = os.path.join(nb_workspace_dir, "data")
generated_dir = os.path.join(nb_workspace_dir, "generated")

os.makedirs(data_dir, exist_ok=True)
os.makedirs(generated_dir, exist_ok=True)

# 1) 메인 노트북 시장 데이터 생성
laptops_data = [
    {"id": "LP-001", "brand": "레노버", "model": "아이디어패드 Slim 3 15IAH8", "price_krw": 599000, "cpu": "Intel Core i5-12450H", "ram_gb": 16, "ssd_gb": 512, "weight_kg": 1.62, "rating": 4.7, "review_count": 420, "category": "가성비/사무용"},
    {"id": "LP-002", "brand": "LG전자", "model": "2026 그램 프로 16", "price_krw": 1890000, "cpu": "Intel Core Ultra 7 155H", "ram_gb": 32, "ssd_gb": 1024, "weight_kg": 1.19, "rating": 4.9, "review_count": 890, "category": "초경량/프리미엄"},
    {"id": "LP-003", "brand": "ASUS", "model": "ROG 제피러스 G14", "price_krw": 2150000, "cpu": "AMD Ryzen 9 8945HS", "gpu": "RTX 4060", "ram_gb": 32, "ssd_gb": 1024, "weight_kg": 1.50, "rating": 4.8, "review_count": 310, "category": "게이밍/크리에이터"},
    {"id": "LP-004", "brand": "한성컴퓨터", "model": "올데이롱 TFX5470UC", "price_krw": 689000, "cpu": "AMD Ryzen 5 7530U", "ram_gb": 16, "ssd_gb": 512, "weight_kg": 1.40, "rating": 4.5, "review_count": 180, "category": "가성비/배터리특화"},
    {"id": "LP-005", "brand": "Apple", "model": "MacBook Air 13 M3", "price_krw": 1590000, "cpu": "Apple M3", "ram_gb": 16, "ssd_gb": 512, "weight_kg": 1.24, "rating": 4.9, "review_count": 1250, "category": "고효율/디자인"}
]
with open(os.path.join(data_dir, "laptops_market_data.json"), "w", encoding="utf-8") as f:
    json.dump(laptops_data, f, ensure_ascii=False, indent=2)

# 2) 다중 파일 인계 실습용 평가 기준 가이드 파일 생성
criteria_data = {
    "scoring_formula": "Value_Score = (RAM_GB * 10 + SSD_GB / 10 + Rating * 20) / (Price_KRW / 100000)",
    "recommendation_threshold": "Top 2 highest Value_Score models",
    "required_columns": ["ID", "Brand", "Model", "Price", "Score", "Category"]
}
with open(os.path.join(data_dir, "specs_criteria.json"), "w", encoding="utf-8") as f:
    json.dump(criteria_data, f, ensure_ascii=False, indent=2)

# 3) 예외 유발용 손상 바이너리 및 백업 카탈로그 생성
with open(os.path.join(data_dir, "corrupted_catalog.bin"), "w", encoding="utf-8") as f:
    f.write("[CORRUPTED_LEGACY_BINARY_BLOB_ERROR_0xDEADBEEF_CANNOT_PARSE]")

with open(os.path.join(data_dir, "backup_laptops_catalog.json"), "w", encoding="utf-8") as f:
    json.dump(laptops_data[:3], f, ensure_ascii=False, indent=2)

print(f"📁 Notebook Dataset Ready at: {nb_workspace_dir}")


📁 Notebook Dataset Ready at: /workspaces/AAWS/artifacts/notebooks


---
## 📌 Part 3. Sub-Agent 래핑 & 정밀 Tool 명세 (3-Layer Protocol Overlay)

Part 1에서 살펴본 **5대 협업 프로토콜**을 실제 코드로 구현하는 핵심 설계 패턴이 바로 **`3-Layer Protocol Overlay`**와 **`InvokeSubAgentInput` 도구 명세**입니다.

### 3.1 왜 `3-Layer Protocol Overlay` 구조인가? (설계 철학)
만약 Worker 에이전트의 `system_prompt`에 *"너는 무조건 5줄 `[TASK REPORT]`로만 대답해야 한다"*는 서브에이전트 규약을 하드코딩해 버리면, 해당 에이전트는 사용자와 직접 대화하는 **단독(Standalone) 모드**로는 재사용할 수 없게 됩니다.

따라서 우리는 Worker의 지침을 **3개의 계층(3-Layer)**으로 분리하여 조립합니다:

```
┌────────────────────────────────────────────────────────────────────────────┐
│ [Layer 1: System Prompt] 전문가 고유 페르소나 (고정)                        │
│  - Market Analyst로서의 도메인 전문성과 분석 역할 정의                      │
│  - 단독(Standalone) 실행과 서브에이전트 실행 모두에서 동일하게 유지          │
├────────────────────────────────────────────────────────────────────────────┤
│ [Layer 2: Protocol Overlay] 서브에이전트 협업 규약 (HumanMessage 상단 주입) │
│  - 🛡️ Prompt Layering: 사담 금지(No-Chat), 정형 포맷 출력 강제              │
│  - 📋 Single Writer 원칙: 공유받은 계획(plan_state.json 등)은 참조만 하고   │
│    직접 수정하지 않으며, 변경이 필요하면 보고서의 Issues에 제안              │
│  - 🔀 Backtracking 규약: 진행 불가 장애 시 즉시 [BLOCKER: 사유] 반환        │
│  - 📄 Artifacts & Summary Pointer: 상세 결과는 파일 저장 후 [TASK REPORT]   │
├────────────────────────────────────────────────────────────────────────────┤
│ [Layer 3: Task Directive] 이번 호출의 구체적 일감 (HumanMessage 하단 주입)  │
│  - ⚡ Dynamic Context Pruning: 부모 대화 이력 없이 target_file_list와       │
│    task_instruction만 슬라이싱하여 전달                                     │
│  - 👶 Self-directed Gathering: 전달받은 파일(데이터 + 공유된 계획 파일)을    │
│    Worker가 스스로 file_read로 열람하며 자율 수행                           │
└────────────────────────────────────────────────────────────────────────────┘
```

### 3.2 5대 협업 프로토콜과 구현 요소의 1:1 매핑
1. **`args_schema` (`InvokeSubAgentInput`)**:
   * 인자별 명확한 타입(`List[str]`, `str`)과 상세한 `Field(description=...)`를 기술하여, 도구 사용법(How)을 도구 스키마에 위임합니다.
2. **⚡ Dynamic Context Pruning (`target_file_list`)**:
   * 부모의 전체 대화 기록을 넘기지 않고, 분석 대상 데이터 파일, 기준표, 그리고 공유할 계획 파일(`plan_state.json` 등)의 경로만 슬라이싱하여 전달합니다.
3. **👶 Self-directed Gathering & Single Writer 원칙**:
   * Worker는 전달받은 파일 목록을 `file_read`로 스스로 열람하여 전체 맥락과 목표를 파악하되, 계획 파일(`plan_state.json`, `task_state.json`)은 **Supervisor만 수정(Single Writer)**하고 Worker는 보고서를 통해 제안만 수행합니다.
4. **🛡️ Prompt Layering (`[SUB-AGENT MODE - STRICT PROTOCOL]`)**:
   * Layer 2 오버레이 헤더를 통해 인사말 없이 즉시 도구를 실행하도록 통제합니다.
5. **📄 Artifacts & Summary Pointer + 🔀 Backtracking (`[TASK REPORT]` / `[BLOCKER]`)**:
   * 상세 리포트는 `file_writer`로 디스크(`artifacts/`)에 기록하고, 부모에게는 5줄 요약 `[TASK REPORT]` 또는 장애 알림 `[BLOCKER]`만 반환합니다.

In [12]:
# 4. Market Analyst Sub-Agent 정의 및 invoke_sub_agent 구축

analyst_llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)

# ─────────────────────────────────────────────────────────────────────────────
# 4.1 Market Analyst Agent 사전 생성 (Layer 1: System Prompt)
# ─────────────────────────────────────────────────────────────────────────────
market_analyst_system_prompt = """You are a Market Analyst Specialist.
Analyze market and product data, compare specifications against criteria, and calculate value scores.
When instructed, inspect assigned files and save detailed reports to designated artifact paths.
Follow all sub-agent execution and reporting protocols strictly without conversational filler."""

worker_middleware = [HierarchicalVisualizerMiddleware(agent_role="Worker (Analyst)", is_subagent=True)]

market_analyst_agent = create_agent(
    model=analyst_llm,
    tools=[file_read, file_writer, file_edit, grep_search],
    system_prompt=market_analyst_system_prompt,
    middleware=worker_middleware
)

# ─────────────────────────────────────────────────────────────────────────────
# 4.2 invoke_sub_agent Pydantic 스키마 및 Protocol Overlay 도구 구현
# ─────────────────────────────────────────────────────────────────────────────
class InvokeSubAgentInput(BaseModel):
    task_instruction: str = Field(
        description="A clear, actionable step-by-step instruction detailing what the worker agent should analyze, calculate, and report."
    )
    target_file_list: List[str] = Field(
        description="List of relative file paths the sub-agent needs to inspect or create (include the Shared Plan File path returned by exit_plan, input data files, and target output artifact path)."
    )
    subagent_role: str = Field(
        default="Market Analyst",
        description="The persona and specialty of the sub-agent (e.g. 'Market Analyst', 'Data Specialist')."
    )

def run_worker_agent(task_instruction: str, target_file_list: List[str], role: str = "Market Analyst") -> str:
    """Sub-Agent를 독립 컨텍스트에서 실행하고 최종 보고서를 반환합니다.
    (Layer 2: Protocol Overlay + Layer 3: Task Directive를 HumanMessage로 주입)
    """
    file_list_str = ", ".join(target_file_list) if target_file_list else "None"
    
    # 🌟 [핵심 패턴] 범용 Sub-Agent Protocol Overlay를 HumanMessage 첫 머리에 주입
    # (특정 도메인 파일명이나 분석 작업을 하드코딩하지 않고, 범용 협업 규약과 Single Writer 원칙만 주입)
    subagent_message = f"""[SUB-AGENT MODE - STRICT PROTOCOL]
You are operating as a sub-agent under a Main Agent. You MUST follow these rules:
1. NO greetings or conversational responses. Execute the task immediately.
2. Context & Plan: If a plan is shared, refer to it to understand the broader context and prior outputs.
   Do NOT modify the plan yourself. If adjustments are needed, propose them in your report.
3. On ANY blocker (corrupted file / format error / missing data / access denied):
   STOP immediately and return EXACTLY:
   [BLOCKER: <concise reason>]
4. On success: write all detailed results to the specified artifact path on disk first, then return EXACTLY:
   [TASK REPORT]
   - Status: SUCCESS
   - Target Files: {file_list_str}
   - Artifacts Created: <created file paths>
   - Summary: <1-2 sentence core finding>
   - Issues: None (or proposed plan adjustments)
══════════════════════════════════════════════════════════
Target Files: {file_list_str}
Instruction: {task_instruction}"""

    # Dynamic Context Pruning: 부모 대화 이력 없이 프로토콜이 결합된 독립 HumanMessage 주입
    result = market_analyst_agent.invoke({"messages": [HumanMessage(content=subagent_message)]})
    return normalize_content(result["messages"][-1].content)


@tool(args_schema=InvokeSubAgentInput)
def invoke_sub_agent(task_instruction: str, target_file_list: List[str], subagent_role: str = "Market Analyst") -> str:
    """Forks an isolated, sandboxed sub-agent with a dedicated worker persona to execute focused discovery or data analysis tasks on target files.

    Use this tool when you need to delegate heavy file reading, multi-file cross-referencing, calculation, or report generation to a specialist worker.

    The sub-agent operates in an isolated ReAct loop, writes detailed artifacts directly to disk, and returns only a concise 5-line summary report.

    PROTOCOL — BEFORE CALLING:
      1. For multi-step tasks, call enter_plan(intent='new' or 'revise', goal=...) -> perform any needed read-only discovery (e.g. glob_search) -> call exit_plan(plan_name, steps) to publish your finalized plan on the Shared Blackboard.
      2. Register a tracked sub-task with task_create() before delegating.
      3. Include the Shared Plan File path (returned by exit_plan) along with input data paths and output artifact paths in target_file_list so the sub-agent sees the full context.

    PROTOCOL — AFTER RECEIVING RESPONSE:
      - [TASK REPORT] + Status: SUCCESS -> Call task_update(task_id, 'COMPLETED').
      - [BLOCKER: <reason>] -> Call task_update(task_id, 'BLOCKED'), call enter_plan(intent='revise', goal=...) to backtrack while preserving blocked task history, inspect alternative files with glob_search, publish the revised plan with exit_plan(plan_name, steps), create a fallback task with task_create(), and re-delegate.

    Returns a formatted `[TASK REPORT]` string on success or `[BLOCKER: <reason>]` on failure.
    """
    block_msg = check_plan_mode_tool_allowed("invoke_sub_agent")
    if block_msg:
        return block_msg
    return run_worker_agent(task_instruction, target_file_list, subagent_role)

print("✅ `invoke_sub_agent` Tool with 3-Layer Protocol Overlay Ready!")

✅ `invoke_sub_agent` Tool with 3-Layer Protocol Overlay Ready!


---
## 📌 Part 4. Supervisor 에이전트 조립 및 [시나리오 1: 정상 가성비 분석] 실행

### 4.1 원칙 중심 오케스트레이션 (Principle-over-Micromanagement)
Supervisor에게 도구 호출 명령(`enter_plan` 호출, `task_create` 호출 등)을 일일이 마이크로매니징하지 않고, **협업 철학과 의사결정 원칙**만 부여합니다:

1. **구조화된 계획 수립 및 맥락 공유**: 복잡한 작업은 계획을 먼저 세우고, 전문가에게 위임할 때는 수립한 계획도 함께 공유하여 전체 맥락을 파악하고 일할 수 있게 합니다.
2. **장애 자가 치유(Backtracking)**: 위임한 작업이 막히더라도(`[BLOCKER]`) 포기하지 않고, 원인을 분석하여 백업 데이터/대안 경로로 자가 치유합니다.
3. **결과 브리핑**: 상세 데이터는 디스크 아티팩트에 보존하고, 사용자에게는 핵심 가치 중심의 요약 브리핑을 제공합니다.

도구 사용법(How)과 인자 규격은 도구의 설명(Docstring) 및 입력 스키마(`args_schema`)에 위임되어 있으므로, LLM은 상위 원칙에 따라 자율적으로 계획 도구와 위임 도구를 조합하여 문제를 해결합니다.

In [13]:
# 5. Supervisor 에이전트 구축 및 시나리오 1 실행
from app.tools.plan import task_board

# 시나리오 1 실행 전 Task Board 및 아카이브 초기화 (클린 상태 보장)
task_board.clear(clear_history=True)  # 현재 세션의 task_state.json / plan_state.json / plan_history.json 초기화

supervisor_llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)

supervisor_system_prompt = """You are a capable AI assistant managing market research and data analysis projects.
Handle straightforward tasks directly, and delegate specialized tasks to expert sub-agents.

═══════════════════════════════════════════════════════════
[Core Principles]
═══════════════════════════════════════════════════════════
1. Structured Planning & Context Sharing:
   - For complex, multi-step tasks, always establish a structured plan on the Task Board before execution.
   - When delegating work to specialist sub-agents, share the established plan so they understand the broader context and objectives.

2. Resilience & Backtracking (Self-Healing):
   - If a delegated task encounters an obstacle ([BLOCKER]), do not stop or give up.
   - Analyze the reported blocker reason, backtrack your plan, and try alternative data or fallback paths to complete the goal.

3. Result Delivery:
   - Ensure all detailed data and reports are saved to designated disk artifacts.
   - Provide the user with a concise, executive briefing highlighting key findings and artifact locations.
"""

# Supervisor 전용 미들웨어 및 도구 장착
supervisor_middleware = [HierarchicalVisualizerMiddleware(agent_role="Supervisor", is_subagent=False)]

supervisor_agent = create_agent(
    model=supervisor_llm,
    tools=[enter_plan, exit_plan, task_create, task_list, task_update, invoke_sub_agent, glob_search],
    system_prompt=supervisor_system_prompt,
    middleware=supervisor_middleware
)

# -----------------------------------------------------------------------------
# [시나리오 1] 정상 노트북 시장 데이터 가성비 분석 요청 실행
# -----------------------------------------------------------------------------
print("🚀 [시나리오 1 시작] 노트북 시장 데이터 가성비 분석 요청...")

user_prompt_1 = """우리 회사의 'artifacts/notebooks/data/laptops_market_data.json' 시장 상품 데이터와 'artifacts/notebooks/data/specs_criteria.json' 기준표를 함께 분석해줘.
Task Board에 계획을 등록하고, Market Analyst에게 분석을 지시한 뒤, 
가격대별 추천 모델 및 비교표가 담긴 상세 보고서를 'artifacts/notebooks/generated/laptop_market_report.md'에 저장하고 최종 브리핑을 해줘."""

response_1 = supervisor_agent.invoke({
    "messages": [HumanMessage(content=user_prompt_1)]
})

print("="*80)
print("👑 [Supervisor 최종 답변]:")
print("="*80)
print(normalize_content(response_1["messages"][-1].content))

🚀 [시나리오 1 시작] 노트북 시장 데이터 가성비 분석 요청...

👑 [Supervisor: Supervisor] 오케스트레이션 루프 시작
  🛠️  [Supervisor Tool Call] ➔ `enter_plan`({"plan_name": "노트북 시장 데이터 및 사양 기준 분석 보고서 작성", "goal": "laptops_market_data.json 및 ...)
  ✅ [Supervisor Tool Result] ➔ `enter_plan`: [PLANNING MODE ENTERED] Session 'default' is now in read-only discovery mode. - Shared Plan File:...
  🛠️  [Supervisor Tool Call] ➔ `glob_search`({"pattern": "artifacts/notebooks/**"})
  ✅ [Supervisor Tool Result] ➔ `glob_search`: Found 7 files matching 'artifacts/notebooks/**' (sorted by most recently modified): artifacts/not...
  🛠️  [Supervisor Tool Call] ➔ `exit_plan`({"steps": ["시장 데이터(laptops_market_data.json) 및 사양 기준표(specs_criteria.json) 분석", "가...)
  ✅ [Supervisor Tool Result] ➔ `exit_plan`: [PLANNING MODE EXITED] Finalized plan '노트북 시장 데이터 및 사양 기준 분석 보고서 작성' with 4 steps (Execution Mode...
  🛠️  [Supervisor Tool Call] ➔ `task_create`({"task_description": "노트북 시장 데이터와 사양 기준표를 분석하여 가격대별 추천 모델 및 비교표가 담긴 상세 보고서(laptop_...)
  ✅ [

In [14]:
# 5.1 시나리오 1 결과 검증 (Task Board 상태 & Artifacts 확인)

print("📋 [Blackboard 상태 확인: plan_state.json + task_state.json]")
print("-" * 50)
print(task_list.invoke({}))

print("\n" + "=" * 50)
print("📄 [Worker가 생성한 Artifact: artifacts/notebooks/generated/laptop_market_report.md]")
print("-" * 50)
report_content = file_read.invoke({"file_path": "artifacts/notebooks/generated/laptop_market_report.md"})
print(report_content)

📋 [Blackboard 상태 확인: plan_state.json + task_state.json]
--------------------------------------------------
{
  "plan_file": "artifacts/users/default/sessions/default/plan/plan_state.json",
  "task_file": "artifacts/users/default/sessions/default/plan/task_state.json",
  "plan": {
    "active": false,
    "phase": "COMPLETED",
    "status": "COMPLETED",
    "plan_name": "노트북 시장 데이터 및 사양 기준 분석 보고서 작성",
    "goal": "laptops_market_data.json 및 specs_criteria.json 분석 후 가격대별 추천 모델 및 비교표 보고서(laptop_market_report.md) 생성",
    "steps": [
      "시장 데이터(laptops_market_data.json) 및 사양 기준표(specs_criteria.json) 분석",
      "가격대별 세그먼트 분류, 사양 대비 가치 평가 및 최적 추천 모델 도출",
      "상세 비교표 및 분석 보고서(laptop_market_report.md) 작성 및 저장",
      "최종 결과 및 주요 인사이트 브리핑"
    ]
  },
  "tasks": [
    {
      "id": 1,
      "description": "노트북 시장 데이터와 사양 기준표를 분석하여 가격대별 추천 모델 및 비교표가 담긴 상세 보고서(laptop_market_report.md) 작성",
      "status": "COMPLETED",
      "created_at": 1795138896.63822,
      "updated_at": 1795138955.4478889

---
## 📌 Part 5. 장애 대응 & [시나리오 2: Backtracking 자가 치유]

### 5.1 Backtracking (백트래킹 자가 치유) 매커니즘
Worker가 샌드박스 내부에서 데이터 손상이나 접근 불가를 마주했을 때:
1. 시스템 전체가 에러로 중단되지 않고 표준 예외 규약인 `[BLOCKER: 원인]` 텍스트를 반환합니다.
2. Supervisor는 이 보고를 수신한 뒤, 실패한 태스크를 `task_update(..., 'BLOCKED')`로 기록하고 **`enter_plan(intent='revise', ...)`로 계획 모드에 재진입**합니다 (`BLOCKED` 이력 보존 + 직전 실행 계획 스냅샷을 `plan_history.json`에 보관).
3. `glob_search` 등으로 대안 경로(`artifacts/notebooks/data/backup_laptops_catalog.json`)를 확인한 뒤 **`exit_plan`으로 수정 계획을 게시**하고, 후속 태스크를 `task_create`로 등록해 2차 위임을 수행함으로써 파이프라인을 자가 치유(Self-Healing)합니다.


In [15]:
# 6. [시나리오 2] 손상 파일 분석 실패 ➔ 백업 카탈로그로 Backtracking 자가 치유 실행
# (주의: 수동 task_board.clear()를 호출하지 않아도 enter_plan(intent='new')가
#  시나리오 1의 완료된 계획/태스크를 plan_history.json에 자동 아카이빙하고 활성 보드를 Task #1부터 리셋합니다.)

print("🚀 [시나리오 2 시작] 손상된 카탈로그 분석 실패 ➔ 백업 데이터로 백트래킹...")

user_prompt_2 = """신규 입고 데이터인 'artifacts/notebooks/data/corrupted_catalog.bin' 파일의 상품 데이터를 'artifacts/notebooks/data/specs_criteria.json' 기준으로 분석해줘.
만약 파일 손상이나 포맷 오류로 분석이 불가능하다면 멈추지 말고 대안 백업 카탈로그를 찾아 백트래킹 분석을 완수한 뒤, 상세 보고서를 'artifacts/notebooks/generated/backup_market_report.md'에 저장하고 브리핑해줘."""

response_2 = supervisor_agent.invoke({
    "messages": [HumanMessage(content=user_prompt_2)]
})

print("="*80)
print("👑 [Supervisor 최종 답변]:")
print("="*80)
print(normalize_content(response_2["messages"][-1].content))

🚀 [시나리오 2 시작] 손상된 카탈로그 분석 실패 ➔ 백업 데이터로 백트래킹...

👑 [Supervisor: Supervisor] 오케스트레이션 루프 시작


  🛠️  [Supervisor Tool Call] ➔ `enter_plan`({"intent": "new", "plan_name": "카탈로그 데이터 스펙 기준 분석 및 백업 백트래킹 계획", "goal": "신규 입고 데이...)
  ✅ [Supervisor Tool Result] ➔ `enter_plan`: [PLANNING MODE ENTERED] Session 'default' is now in read-only discovery mode. - Shared Plan File:...
  🛠️  [Supervisor Tool Call] ➔ `glob_search`({"pattern": "artifacts/notebooks/**/*"})
  ✅ [Supervisor Tool Result] ➔ `glob_search`: Found 7 files matching 'artifacts/notebooks/**/*' (sorted by most recently modified): artifacts/n...
  🛠️  [Supervisor Tool Call] ➔ `exit_plan`({"plan_name": "신규 카탈로그 분석 및 대안 백업 데이터 백트래킹 분석", "steps": ["신규 입고 데이터(corrupted_cat...)
  ✅ [Supervisor Tool Result] ➔ `exit_plan`: [PLANNING MODE EXITED] Finalized plan '신규 카탈로그 분석 및 대안 백업 데이터 백트래킹 분석' with 4 steps (Execution Mo...
  🛠️  [Supervisor Tool Call] ➔ `task_create`({"task_description": "신규 입고 데이터(corrupted_catalog.bin)의 specs_criteria.json 기준 분석 ...)
  ✅ [Supervisor Tool Result] ➔ `task_create`: [TASK CREATED] Task #1: '신규 입고 데이터(co

In [16]:
print("\n" + "="*50)
print("📋 [자가 치유 완료된 Blackboard 최종 상태]")
print("="*50)
print(task_list.invoke({}))

print("\n" + "="*50)
print("🗂️ [plan_history.json 아카이브 요약]")
print("="*50)
for idx, item in enumerate(get_plan_history(), start=1):
    plan_name = item.get("plan", {}).get("plan_name", "")
    reason = item.get("reason", "")
    statuses = [f"#{t['id']}:{t['status']}" for t in item.get("tasks", [])]
    print(f"{idx}. [{reason}] {plan_name} -> {statuses}")


📋 [자가 치유 완료된 Blackboard 최종 상태]
{
  "plan_file": "artifacts/users/default/sessions/default/plan/plan_state.json",
  "task_file": "artifacts/users/default/sessions/default/plan/task_state.json",
  "plan": {
    "active": false,
    "phase": "COMPLETED",
    "status": "COMPLETED",
    "plan_name": "백업 카탈로그 기반 백트래킹 분석 및 최종 보고서 작성",
    "goal": "corrupted_catalog.bin 파싱 불가에 따른 백업 카탈로그(backup_laptops_catalog.json) 백트래킹 분석 및 보고서 작성",
    "steps": [
      "대안 백업 카탈로그(backup_laptops_catalog.json) 데이터 파싱 및 specs_criteria.json 기준 분석",
      "상세 분석 결과를 'artifacts/notebooks/generated/backup_market_report.md'에 마크다운 보고서로 작성 및 저장",
      "최종 브리핑 제공"
    ]
  },
  "tasks": [
    {
      "id": 1,
      "description": "신규 입고 데이터(corrupted_catalog.bin)의 specs_criteria.json 기준 분석 시도",
      "status": "BLOCKED",
      "created_at": 1795138986.6264162,
      "updated_at": 1795139003.51529
    },
    {
      "id": 2,
      "description": "백업 카탈로그(backup_laptops_catalog.json)를 specs_criteria.json 기준으로 분석하고 bac

---
## 📌 Part 6. 실무 요약 & AAWS 프로덕션 확장 로드맵

### 💡 정보 불균형 극복을 위한 멀티에이전트 설계 3대 황금률
1. **대화로 소통하지 말고 구조화된 Task Board로 소통하라 (Blackboard Pattern & Single Writer)**:
   * `enter_plan`, `exit_plan`, `task_create`, `task_update`를 통해 세션 단절과 환각 없이 작업 진행도를 투명하게 통제합니다.
   * 하위 에이전트는 공유받은 계획을 읽어 전체 맥락을 파악하되, 직접 계획표를 수정하지 않고 보고서를 통해 제안만 수행합니다.
2. **세부 결과는 `artifacts/`에 영구 보존하고, 부모에게는 요약 포인터만 반환하라**:
   * 부모의 컨텍스트를 오염시키지 않아 수십 번의 태스크 반복에도 토큰 비용이 선형 증가하지 않습니다.
3. **Worker에게는 3-Layer Protocol Overlay로 침묵 실행(No-Chat)과 Backtracking 규약을 강제하라**:
   * 전문가 고유 페르소나(Layer 1)와 서브에이전트 협업 규약(Layer 2)을 분리하여 재사용성을 높이고, 실패 시 핑퐁 대화 대신 `[BLOCKER]`를 반환하여 부모가 전략을 동적으로 재설계할 수 있게 합니다.

---

### 🚀 AAWS 프로덕션 확장 안내 (`app/agents/supervisor/`)

우리가 실습에서 구현한 5대 협업 프로토콜과 3-Layer 아키텍처는 AAWS 프로덕션 환경에 다음과 같이 1:1로 확장 적용됩니다:

```
👑 Supervisor Agent (app/agents/supervisor/)
 ├── 🔌 Logging & Visualizer Middleware
 ├── 🛠️ Planning Tools (enter_plan, task_create, task_list, task_update, exit_plan)
 ├── 🛠️ Orchestration Tools (list_sub_agents, invoke_sub_agent, get_sub_agent_job_status)
 ├── 🕷️ The Scraper (app/agents/scraper/)
 │    └── 사이트 탐색 + 코드 작성/실행 ➔ artifacts/에 데이터 & extraction_plan.json 저장
 └── 📊 The Analyst (app/agents/analyst/)
      └── 수집 데이터 통계 분석 + 시각화 차트 ➔ artifacts/에 종합 분석 보고서 저장
```